# Solutions · 13 · Reverse osmosis and ultrafiltration

Worked solutions to the exercises of [notebook 13](../notebooks/13_reverse_osmosis_and_ultrafiltration.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engsep is missing): install it from GitHub
    import engsep
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engsep"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engsep import membranes as mb
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

A brackish-water plant (2 g/L) runs at 15 bar with 75 % recovery. Compute the concentrate's osmotic pressure and check that the pressure suffices; then find the permeate concentration at the feed end and at the concentrate end of the module and their flow-weighted average.

In [2]:
T, c_feed, recovery, dP = 298.15, 2 / 58.44, 0.75, 15e5
c_conc = c_feed / (1 - recovery)
pi_conc = mb.osmotic_pressure(2 * c_conc, T)
print(f"concentrate {c_conc*58.44:.1f} g/L, osmotic pressure {pi_conc/1e5:.2f} bar against 15 bar applied: {'sufficient' if pi_conc < dP else 'insufficient'}")
A_w, B_s = 6e-12, 5e-8
ro_in = mb.ro_permeate_concentration(A_w, B_s, dP, c_feed, T)
ro_out = mb.ro_permeate_concentration(A_w, B_s, dP, c_conc, T)
w_in, w_out = ro_in["J_w"], ro_out["J_w"]
c_avg = (w_in * ro_in["c_permeate"] + w_out * ro_out["c_permeate"]) / (w_in + w_out)
print(f"permeate at the feed end {ro_in['c_permeate']*58.44*1000:.0f} mg/L, at the concentrate end {ro_out['c_permeate']*58.44*1000:.0f} mg/L; flux-weighted average about {c_avg*58.44*1000:.0f} mg/L")

concentrate 8.0 g/L, osmotic pressure 6.79 bar against 15 bar applied: sufficient
permeate at the feed end 12 mg/L, at the concentrate end 80 mg/L; flux-weighted average about 38 mg/L


Brackish water is easy: the concentrate at 75 % recovery has an osmotic pressure of a few bar, so 15 bar
leaves ample driving force; the permeate gets saltier along the module as the feed concentrates and the
water flux falls, and the product is the flux-weighted blend - four times the salt of the feed-end permeate.

## Exercise 2

Energy: compute the pumping energy per m3 of permeate for seawater at 60 bar and 45 % recovery with and without an energy-recovery device that returns 95 % of the concentrate's pressure energy. Compare with the thermodynamic minimum.

In [3]:
dP, recovery, rho, eta_pump = 60e5, 0.45, 1025.0, 0.85
feed_per_perm = 1 / recovery
E_pump = dP * feed_per_perm / eta_pump / 3.6e6                       # kWh per m3 of permeate, pressurising the whole feed
conc_per_perm = feed_per_perm - 1
E_recovered = 0.95 * dP * conc_per_perm / 3.6e6
pi_avg = mb.osmotic_pressure(2 * 35 / 58.44 / (1 - recovery / 2), 298.15)
E_min = pi_avg / 3.6e6
print(f"without energy recovery: {E_pump:.2f} kWh/m3; with 95 % recovery from the concentrate: {E_pump - E_recovered:.2f} kWh/m3; thermodynamic minimum (osmotic pressure of the average concentrate): {E_min:.2f} kWh/m3")

without energy recovery: 4.36 kWh/m3; with 95 % recovery from the concentrate: 2.42 kWh/m3; thermodynamic minimum (osmotic pressure of the average concentrate): 1.06 kWh/m3


Pressurising 2.2 m$^3$ of seawater to make 1 m$^3$ of permeate costs about 4 kWh without energy recovery;
recovering the pressure energy of the concentrate halves it, within a factor of two of the thermodynamic
minimum. Energy-recovery devices are what made seawater desalination affordable.

## Exercise 3

**Implement it yourself:** iterate the RO permeate concentration yourself (start with c_p = 0, compute Delta pi, J_w, J_s, update c_p) and reproduce `membranes.ro_permeate_concentration` at 60 bar.

In [4]:
A_w, B_s, dP, c_feed, T = 3e-12, 2e-8, 60e5, 35 / 58.44, 298.15
c_p = 0.0
for it in range(50):
    d_pi = 2 * (c_feed - c_p) * 1000 * 8.314462618 * T
    J_w = A_w * (dP - d_pi); J_s = B_s * (c_feed - c_p)
    c_new = J_s / J_w
    if abs(c_new - c_p) < 1e-12:
        break
    c_p = c_new
print(f"by hand: c_p = {c_p*58.44*1000:.2f} mg/L after {it} iterations; library: {mb.ro_permeate_concentration(A_w, B_s, dP, c_feed, T)['c_permeate']*58.44*1000:.2f}")

by hand: c_p = 76.66 mg/L after 4 iterations; library: 76.66


The permeate concentration is a fixed point of the two flux equations; a few iterations converge to the
library's value.